In [ ]:
import numpy as np
from tqdm import tqdm               # to show progess bar
import matplotlib.pyplot as plt     # for showing images
from PIL import Image               # resizing the image and converting to gray scale
import os                           # for loading images from a given path
from random import shuffle          # for shuffing the images

In [ ]:
# floder paths which contain training image and testing images
TRAIN_DIR = '/content/traning'
TEST_DIR = '/content/testing'

In [ ]:
# we will resixze all images with PIL to a dimension of 120 * 120
IMG_SIZE = 120
MODEL_NAME = 'NIKEvsADIDAS'

In [ ]:
# take img name
# this function takes image name as parameter
# if image name contain ADIDAS it returns [0,1]
# if image name contain Nike it returns [1,0]
# so that we can feed to neural network
def label_image(img):
  word_label = img.split('_')[0]
  if word_label == 'NIKE': return [1,0]
  elif word_label == 'ADIDAS': return [0,1]

In [ ]:
def create_train_data():
  train_data = []
  for img in tqdm(os.listdir(TRAIN_DIR)):
    path = os.path.join(TRAIN_DIR,img)
    if os.path.isfile(path):
      label = label_image(img)

      # PIL
      img = Image.open(path)
      img = img.convert('L')
      img = img.resize((IMG_SIZE, IMG_SIZE), Image.Resampling.LANCZOS)
      #Create composite image by blending images using a transparency mask...... Image
      # ANTIALIAS (a high-quality downsampling filter).
      train_data.append([np.array(img), np.array(label)])
  shuffle(train_data)
  #np.save('train_data.npy',train_data)
  return train_data

In [ ]:
#ignore this
def process_test_data():
  test_data = []
  for img in tqdm(os.listdir(TEST_DIR)):
    path = os.path.join(TEST_DIR,img)
    if "DS_Store" not in path:
      img_num = img.split('_')[1]
      img = Image.open(path)
      img = img.convert('L')
      img = img.resize((IMG_SIZE, IMG_SIZE), Image.Resampling.LANCZOS)
      test_data.append([np.array(img), img_num])
  shuffle(test_data)
  #np.save('test_data.npy', test_data)
  return test_data

In [ ]:
train_data = create_train_data()


FileNotFoundError: [Errno 2] No such file or directory: '/content/traning'

In [ ]:
plt.imshow(train_data[1][0], cmap='gist_gray')
plt.show()

In [ ]:
# Buliding the Network

In [ ]:
import keras
from keras.layers import Conv2D, MaxPooling2D
from keras.models import Sequential
from keras.layers import Dense, Dropout, Activation, Flatten  # how many activation functions are there and work process

In [ ]:
model = keras.Sequential()
model.add(Conv2D(16,3,activation='relu', input_shape = (120,120,1)))       # input_shape = (120,120,1) input layer 1 is grayscale
model.add(MaxPooling2D(pool_size=(2,2)))                                            # each convalutional should have relu
                                                                                    # add is used to adding a layer
model.add(Conv2D(32,3,activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2))) # (2,2) is window size                     # each cell contain 0 to 255 values

model.add(Conv2D(64,3,activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))

model.add(Flatten())

model.add(Dense(64, activation='relu'))
model.add(Dropout(0.2))                                      # class room project (poc) we have less amount of data
model.add(Dense(2, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# what is accuracy -> how many your predicting correctly
# what is loss -> how closely is your preciction
# softmax is tell probability
# who calculate the loss -> categorical crossentropy will check the loss
# adam optimiser changes weights based on gradient

In [ ]:
x = np.array([i[0] for i in train_data], dtype = np.float32).reshape(-1, IMG_SIZE, IMG_SIZE, 1)
x = x/255.0
y = np.array([i[1] for i in train_data])

In [ ]:
model.fit(x, y, epochs= 10)